# 모델 실험 템플릿

팀 공통 실험 노트북입니다. **이 파일은 고치지 말고 복사해서** 씁니다.
각자 모델 하나(`yolo11m.pt` 또는 `yolo26m.pt`)를 맡아 옵티마이저·lr0 등을 조정합니다.

1. `notebooks/exp_<이름>.ipynb`로 복사합니다.
2. 설정 셀의 **내 실험** 부분만 바꿉니다. **팀 공통** 값은 모두 같아야 서로 점수를 비교할 수 있으므로 바꾸지 않습니다.
3. 순서
   1. 로컬(Mac)에서 `SMOKE = True`로 동작 확인 (epoch 1, 학습 사진 10%, wandb 끔)
   2. Colab GPU에서 `SMOKE = False`, `STAGE = "screen"`: fold 0만 학습해 설정을 고릅니다. 다른 설정과는 fold 0 점수끼리 비교합니다.
   3. 고른 설정은 `STAGE = "confirm"`: 5 fold를 모두 학습합니다. fold 0은 screen 결과를 그대로 씁니다. **모델 비교는 이 5 fold 평균·표준편차로** 합니다.
4. 실험 이름은 설정으로 자동으로 만듭니다 (예: `a_yolo26m_AdamW_lr0.001_cos`). 같은 설정을 다시 돌리면 끝난 fold는 건너뛰고, 끊긴 fold는 `last.pt`에서 이어 학습합니다.
5. 결과는 wandb 팀 프로젝트에 `<실험 이름>/<run>` 이름으로 쌓이고, config에 옵티마이저·lr0 등이 남습니다.
   - Colab: 커널 선택 → **Colab** → GPU 런타임. VS Code 탐색기에서 `.env`를 우클릭 → **Upload to Colab** (`DATA_URL`, `WANDB_API_KEY`, `WANDB_ENTITY`).
   - Colab은 `BRANCH`의 코드를 받아 돌리므로, 로컬에서 고친 `.py`는 push한 뒤 실행합니다.
6. 커밋할 때는 출력을 지웁니다.

## 1. 설정

실험마다 **내 실험**과 **실행** 부분만 바꿉니다. 팀 공통 값을 바꿔야 할 일이 생기면 팀에서 정한 뒤 템플릿을 고칩니다.

In [1]:
# ===== 팀 공통: 바꾸지 않는다 (AGENTS.md 기본값) =====
LABEL = "v1"
SPLIT = "setup30fix_v1_s42"
EPOCHS = 100
IMGSZ = 640
SEED = 42
AUG = "on"  # models/train.py의 AUG_PRESETS
FRACTION = 1.0

# ===== 내 실험 =====
MEMBER = "kwon"  # 이름이나 이니셜. 실험 이름 맨 앞에 붙는다
MODEL = "yolo11m.pt"  # yolo11m.pt 또는 yolo26m.pt
# SGD, MuSGD, Adam, Adamax, AdamW, NAdam, RAdam, RMSProp. auto는 lr0를 무시하므로 쓰지 않는다
OPTIMIZER = "AdamW"
LR0 = 1e-3  # None이면 SGD 계열 0.01, 그 밖 0.001
COS_LR = False  # True면 학습률을 cosine 곡선으로 줄인다
BATCH = 16  # 팀 기본 16. GPU 메모리가 부족하면 줄인다
# 그 밖 train() 인자. 괄호 안은 ultralytics 기본값
# lrf(0.01), momentum(0.937), weight_decay(0.0005), warmup_epochs(3.0), patience(100)
TRAIN_KWARGS = {}  # 예: {"weight_decay": 0.001, "patience": 30}
TAG = "#1"  # 이름이 같아지는 실험을 구분할 짧은 꼬리표 (선택)

# ===== 실행 =====
SMOKE = True  # True면 동작 확인만. Colab GPU 학습은 False
STAGE = "screen"  # screen: fold 0만, confirm: 5 fold 전부
BRANCH = "main"  # Colab에서 받을 브랜치

## 2. 환경

Colab이면 저장소를 받고 패키지를 설치한 뒤 결과 폴더를 Drive에 둡니다. 로컬에서는 프로젝트 루트를 import 경로에 넣기만 합니다.

In [2]:
import os
import sys
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").is_dir()
RUNS_DIR = None  # None이면 프로젝트의 runs/

if IN_COLAB:
    # colab.ipynb 2–4단계와 같다: 저장소 받기, uv.lock 버전 설치, .env 읽기
    import importlib

    REPO_DIR = Path("/content/pill-bbox")
    if REPO_DIR.exists():
        !git -C {REPO_DIR} fetch -q origin {BRANCH}
        !git -C {REPO_DIR} checkout -q {BRANCH}
        !git -C {REPO_DIR} pull -q
    else:
        !git clone -q -b {BRANCH} https://github.com/def-uk/pill-bbox.git {REPO_DIR}
    os.chdir(REPO_DIR)
    if not Path("~/.local/bin/uv").expanduser().exists():
        !curl -LsSf https://astral.sh/uv/install.sh | sh
    os.environ["PATH"] = f"{Path('~/.local/bin').expanduser()}:{os.environ['PATH']}"
    # torch는 lock에 적힌 CUDA 12.6 빌드로 설치한다
    req = "/tmp/requirements.txt"
    torch_index = "--extra-index-url https://download.pytorch.org/whl/cu126"
    strategy = "--index-strategy unsafe-best-match"
    !uv export -q --frozen --no-dev --no-hashes --no-emit-project -o {req}
    !uv pip install -q --python {sys.executable} -r {req} {torch_index} {strategy}
    sys.path.insert(0, str(REPO_DIR / "notebooks"))
    importlib.invalidate_caches()

    from dotenv import load_dotenv

    load_dotenv("/content/.env")  # 이미 설정된 환경 변수가 우선한다
    for name in ["DATA_URL", "WANDB_API_KEY", "WANDB_ENTITY"]:
        if not os.getenv(name):
            try:
                from google.colab import userdata

                os.environ[name] = userdata.get(name)
            except Exception:
                pass
    if not os.getenv("WANDB_API_KEY"):
        os.environ["WANDB_MODE"] = "disabled"
        print("WANDB_API_KEY가 없어 wandb를 끄고 진행합니다.")

    # 결과를 Drive에 두어야 런타임이 끊겨도 남는다
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        RUNS_DIR = Path("/content/drive/MyDrive/pill-bbox/runs")
    except Exception as e:
        print(f"Drive 연결 실패({e}). 결과가 런타임에만 남으니 끊기기 전에 내려받으세요.")
    !git branch --show-current && git log --oneline -1

# 프로젝트 루트를 import 경로에 넣는다
import nb_data  # noqa: E402, F401

downloading uv 0.13.6 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Drive 연결 실패(mount failed). 결과가 런타임에만 남으니 끊기기 전에 내려받으세요.
main
79e2559 (HEAD -> main, origin/main, origin/HEAD) feat: 팀 공통 모델 실험 노트북 템플릿 추가 (#63)


## 3. 실험 준비

설정을 검사하고 실험 이름을 만듭니다. 같은 학습을 터미널에서 다시 돌리는 `model_main` 명령을 출력하고, 설정은 `<RUNS_DIR>/<exp>/notebook_config.json`에 남깁니다.

In [3]:
import json
from datetime import datetime

from data_process.split import N_FOLDS
from models.train import default_lr0
from src.config import PROJECT_ROOT, get_device
from src.config import RUNS_DIR as DEFAULT_RUNS_DIR

# 가중치 이름(yolo11m.pt)은 현재 폴더에 받아지므로 루트로 옮겨 팀 공통 위치에 둔다
os.chdir(PROJECT_ROOT)
RUNS_DIR = RUNS_DIR or DEFAULT_RUNS_DIR
DEVICE = get_device()

if not MEMBER:
    raise ValueError("MEMBER에 이름이나 이니셜을 적으세요.")
if MODEL not in ["yolo11m.pt", "yolo26m.pt"]:
    raise ValueError(f"후보 모델은 yolo11m.pt, yolo26m.pt입니다: {MODEL}")
if OPTIMIZER == "auto":
    raise ValueError("auto는 lr0를 무시합니다. 옵티마이저를 직접 고르세요 (예: AdamW).")
if STAGE not in ["screen", "confirm"]:
    raise ValueError(f"STAGE는 screen 또는 confirm입니다: {STAGE}")
lr0 = default_lr0(OPTIMIZER) if LR0 is None else LR0

# 결과 폴더 이름에는 lr0 등이 없으므로, 내 실험 설정을 모두 실험 이름에 넣어 구분한다
parts = [MEMBER, Path(MODEL).stem, OPTIMIZER, f"lr{lr0:g}"]
parts += ["cos"] if COS_LR else []
parts += [] if BATCH == 16 else [f"b{BATCH}"]
parts += [
    f"{key}{value:g}" if isinstance(value, float) else f"{key}{value}"
    for key, value in sorted(TRAIN_KWARGS.items())
]
parts += [TAG] if TAG else []
exp_name = "_".join(parts)

# 동작 확인이면 짧은 값을 쓰고, 실험 이름에 시각을 붙여 이전 확인과 겹치지 않게 한다
if SMOKE:
    exp, epochs, fraction, folds = f"smoke_{exp_name}_{datetime.now():%m%d_%H%M}", 1, 0.1, [0]
    os.environ["WANDB_MODE"] = "disabled"
else:
    if DEVICE != "cuda":
        raise RuntimeError(f"본 학습은 GPU에서 합니다(지금 {DEVICE}). 동작 확인은 SMOKE = True.")
    exp, epochs, fraction = exp_name, EPOCHS, FRACTION
    folds = [0] if STAGE == "screen" else list(range(N_FOLDS))

# 같은 학습을 터미널에서 다시 돌릴 수 있게 model_main 명령으로 남긴다
cmd = ["uv run python -m models.model_main", "--model", MODEL, "--label", LABEL]
cmd += ["--split", SPLIT, "--folds", *map(str, folds), "--epochs", str(epochs), "--exp", exp]
cmd += ["--imgsz", str(IMGSZ), "--batch", str(BATCH), "--seed", str(SEED)]
cmd += ["--fraction", str(fraction), "--aug", AUG, "--optimizer", OPTIMIZER, "--lr0", f"{lr0:g}"]
cmd += ["--cos-lr"] if COS_LR else []
cmd += [] if RUNS_DIR == DEFAULT_RUNS_DIR else ["--runs-dir", str(RUNS_DIR)]
cmd += ["--resume"]
command = " ".join(cmd)

config = {
    "exp": exp,
    "member": MEMBER,
    "stage": STAGE,
    "smoke": SMOKE,
    "device": DEVICE,
    "model": MODEL,
    "label": LABEL,
    "split": SPLIT,
    "folds": folds,
    "epochs": epochs,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "seed": SEED,
    "fraction": fraction,
    "aug": AUG,
    "optimizer": OPTIMIZER,
    "lr0": lr0,
    "cos_lr": COS_LR,
    "train_kwargs": TRAIN_KWARGS,
    "command": command,
    "created_at": datetime.now().isoformat(timespec="seconds"),
}
exp_dir = RUNS_DIR / exp
exp_dir.mkdir(parents=True, exist_ok=True)
(exp_dir / "notebook_config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"실험 {exp} ({STAGE}, fold {folds}), 장치 {DEVICE}")
print(f"결과 {exp_dir}")
print(command)
if TRAIN_KWARGS:
    print(f"추가 train() 인자 (CLI에 없음): {TRAIN_KWARGS}")

실험 smoke_kwon_yolo11m_AdamW_lr0.001_#1_1028_0510 (screen, fold [0]), 장치 cuda
결과 /content/pill-bbox/runs/smoke_kwon_yolo11m_AdamW_lr0.001_#1_1028_0510
uv run python -m models.model_main --model yolo11m.pt --label v1 --split setup30fix_v1_s42 --folds 0 --epochs 1 --exp smoke_kwon_yolo11m_AdamW_lr0.001_#1_1028_0510 --imgsz 640 --batch 16 --seed 42 --fraction 0.1 --aug on --optimizer AdamW --lr0 0.001 --resume


## 4. 데이터

데이터는 `data_process.data_main`으로만 만듭니다. fold마다 train/val 사진 수입니다.

In [4]:
from nb_data import ensure_pipeline, load_folds

# 라벨·fold·data.yaml이 없으면 data_main을 실행해 만든다 (처음 약 25초, 원본이 없으면 받기부터)
ensure_pipeline(LABEL, [SPLIT])
load_folds(SPLIT).groupby(["fold", "split"]).size().unstack()

split,train,val
fold,,
0,189,43
1,196,36
2,206,26
3,202,30
4,204,28


## 5. 학습과 평가

`best.pt`는 검증 mAP75-95가 가장 높은 epoch입니다. 런타임이 끊기면 위에서부터 다시 실행하면 이어서 학습합니다.

In [5]:
from models.evaluate import MAIN_METRIC, evaluate_fold, summarize_exp
from models.train import train

# 실험 이름이 설정에서 나오므로 같은 이름 = 같은 설정이다. 그래서 항상 resume으로 돌린다:
# 끝난 fold(screen의 fold 0 등)는 건너뛰고, 끊긴 fold는 last.pt에서 이어 간다.
for fold in folds:
    save_dir = train(
        MODEL,
        LABEL,
        SPLIT,
        fold,
        epochs=epochs,
        exp=exp,
        imgsz=IMGSZ,
        batch=BATCH,
        seed=SEED,
        fraction=fraction,
        aug=AUG,
        optimizer=OPTIMIZER,
        lr0=lr0,
        cos_lr=COS_LR,
        runs_dir=RUNS_DIR,
        resume=True,
        **TRAIN_KWARGS,
    )
    _, summary = evaluate_fold(
        save_dir / "weights" / "best.pt",
        LABEL,
        SPLIT,
        fold,
        exp=exp,
        imgsz=IMGSZ,
        batch=BATCH,
        runs_dir=RUNS_DIR,
    )
    print(f"{save_dir.name}: {MAIN_METRIC} {summary[MAIN_METRIC]:.4f}")

New https://pypi.org/project/ultralytics/8.4.191 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.182 🚀 Python-3.13.16 torch-2.14.1+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/pill-bbox/data/processed/yolo/v1/splits/setup30fix_v1_s42/fold_0/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=0.1, freeze=None, hsv_h=0.005, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det

## 6. 결과 요약

다른 사람·다른 설정과 비교할 때 screen은 fold 0끼리, 모델 비교는 confirm의 5 fold 평균·표준편차로 합니다.

In [6]:
# fold 평균·표준편차. <exp>/results/summary.csv에도 저장된다. screen은 fold 하나라 std가 비어 있다.
summary = summarize_exp(exp, RUNS_DIR)
cols = [("n_folds", "count"), (MAIN_METRIC, "mean"), (MAIN_METRIC, "std")]
cols += [("map50", "mean"), ("map50_95", "mean")]
cols += [(name, "mean") for name in ["precision", "recall", "mean_iou", "recall_≤5"]]
summary[cols].round(4)

KeyError: "[('precision', 'mean'), ('recall', 'mean'), ('mean_iou', 'mean'), ('recall_≤5', 'mean')] not in index"

## 7. 학습 곡선

fold별 매 epoch 학습·검증 loss(윗줄)와 검증 mAP(아랫줄)입니다. 같은 그림이 run 폴더의 `results.png`로도 저장됩니다.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import MaxNLocator

from models.train import MAIN_METRIC_KEY

METRIC_KEYS = ["metrics/mAP50(B)", "metrics/mAP50-95(B)", MAIN_METRIC_KEY]

# ultralytics가 매 epoch 쓰는 results.csv를 fold별로 읽는다 (run 폴더 이름 끝이 _fold<k>)
histories = {}
for path in sorted(exp_dir.glob("*/results.csv")):
    history = pd.read_csv(path)
    history.columns = history.columns.str.strip()
    histories[path.parent.name.rsplit("_", 1)[-1]] = history

if not histories:
    print(f"{exp_dir}에 results.csv가 없습니다. 학습 셀을 먼저 실행하세요.")
else:
    first = next(iter(histories.values()))
    loss_names = [c.removeprefix("train/") for c in first.columns if c.startswith("train/")]
    n_cols = max(len(loss_names), len(METRIC_KEYS))
    fig, axes = plt.subplots(2, n_cols, figsize=(4.5 * n_cols, 7), squeeze=False)
    for i, (fold, history) in enumerate(histories.items()):
        color, epoch = f"C{i}", history["epoch"]
        # 윗줄: loss. 실선은 train, 점선은 val. val이 오르는데 train만 내려가면 과적합이다
        for ax, name in zip(axes[0], loss_names, strict=False):
            ax.plot(epoch, history[f"train/{name}"], color=color, marker=".", label=f"{fold} train")
            ax.plot(epoch, history[f"val/{name}"], color=color, linestyle="--", label=f"{fold} val")
            ax.set_title(name)
        # 아랫줄: 검증 mAP. 별은 best.pt를 고른 epoch (mAP75-95 최고)
        for ax, key in zip(axes[1], METRIC_KEYS, strict=False):
            ax.plot(epoch, history[key], color=color, marker=".", label=fold)
            ax.set_title(key.removeprefix("metrics/").removesuffix("(B)"))
        best = history[MAIN_METRIC_KEY].idxmax()
        axes[1, len(METRIC_KEYS) - 1].plot(
            epoch[best], history.loc[best, MAIN_METRIC_KEY], "*", color=color, markersize=14
        )
    for ax in axes.ravel():
        ax.set_xlabel("epoch")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    for ax in [*axes[0, len(loss_names) :], *axes[1, len(METRIC_KEYS) :]]:
        ax.set_visible(False)
    axes[0, 0].legend(fontsize=7)
    axes[1, 0].legend(fontsize=7)
    fig.suptitle(exp)
    fig.tight_layout()
    plt.show()

## 실험 기록

- **실험 이름 / 단계**: (설정 셀 출력의 이름, screen 또는 confirm)
- **이 설정을 고른 이유**:
- **명령**: 실험 준비 셀이 출력한 `uv run python -m models.model_main ...`
- **결과**: screen이면 fold 0 mAP75-95, confirm이면 5 fold 평균 ± 표준편차
- **결론 / 다음 실험**: 